In [1]:
import pickle
import numpy as np
import os
import tqdm
import torch
import gc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2023-11-20 01:15:05.694393: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/shevkunov/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


matrix_approx_zeshel.py


# Open Data loader & context

In [2]:
def load_ment_to_ent_scores(directory = "yugioh", shuffle_rows = 0, full = True):
    data = list()

    for file in os.listdir(directory):
        path = f"{directory}/{file}"
        print(f"Loading file {path}")
        with open(path, "rb") as f:
            data.append(
                pickle.load(f)
            )
    data = sorted(data, key = lambda x: x["arg_dict"]["n_ment_start"])

    for i in range(len(data) - 1):
        if full:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] == data[i + 1]["arg_dict"]["n_ment_start"]
        else:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] <= data[i + 1]["arg_dict"]["n_ment_start"]
        
    ment_to_ent_scores = list(map(lambda x: x["ment_to_ent_scores"], data))
    ment_to_ent_scores = np.vstack(ment_to_ent_scores)
    print("Loaded shape = ", ment_to_ent_scores.shape)
    
    if shuffle_rows:
        print(f"Shuffling... (seed = {shuffle_rows})")
        np.random.seed(shuffle_rows)
        np.random.shuffle(ment_to_ent_scores)
    
    return ment_to_ent_scores

In [33]:
from sklearn.cluster import KMeans

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, shuffle=False, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        if shuffle:
            np.random.shuffle(self.relevs)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:100]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def get_kmeans_games(self, all_from_labels=True):
        X = self.get_relevs("train").T

        k_func = (
            (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
            if not all_from_labels else
            (lambda C : from_labels(X, C.labels_))
        )
        K_KMeans = k_func(
            KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
        )
        
        return K_KMeans
    
    def set_kmeans_games_as_key(self, *args, **kwargs):
        self.key_games = self.get_kmeans_games(*args, **kwargs)
        return self
    

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

# Games Data loader & context

In [4]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            
            self.key_size = key_size
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def get_kmeans_games(self, all_from_labels=True):
            X = self.get_relevs("train").T

            k_func = (
                (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
                if not all_from_labels else
                (lambda C : from_labels(X, C.labels_))
            )
            K_KMeans = k_func(
                KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
            )

            return K_KMeans

        def set_kmeans_games_as_key(self, *args, **kwargs):
            self.key_games = self.get_kmeans_games(*args, **kwargs)
            return self

        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id]
                    for r_i in r_i_array[:100]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

# Models

In [5]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        self.trus_top = dict()
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):    
        recs = self.recommend(t)
        
        if isinstance(recs, list):
            recs = np.array(recs)
        
        mse = np.mean((recs - self.get_user_scores(t)) ** 2)
        
        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :topsize].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :topsize]
        
        if (t, topsize) not in self.trus_top:
            trus = self.get_user_scores(t)
            trus = np.argsort(-trus, axis=1)[:, :topsize]
            self.trus_top[(t, topsize)] = trus
            
        trus = self.trus_top[(t, topsize)]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [6]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False,
    "Winit": "glorot0.01",
    "Wfreeze": False,
    "TEinit": "legacy",
    "normalize_embs": True
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in sorted(ctx.key_games)])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["TEinit"] == "legacy":
                value = self.embed_games
            elif self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                value = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print(type(value))
            else:
                assert False, "unknown TEinit: " + str(self.fit_kwargs["TEinit"])
                
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(value, dtype=tf.float32),
                trainable=True
            )
        else:
            if self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                self.embed_games = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print("ANNCur init")
            
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        if self.fit_kwargs["normalize_embs"]:
            return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        else:
            return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["normalize_embs"]:
                return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
            else:
                return self.embed_games 
        else:
            return self.trainable_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        
        
        if self.p["Winit"] == "glorot0.01":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
            values = values / 100.  
        elif self.p["Winit"] == "glorot":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        elif self.p["Winit"] == "eye":
            values = np.eye(train_user_embs.shape[1], game_embs.shape[1])
        else:
            assert False, "init is not known:" + str(self.p["init"])
            
        self.W = tf.Variable(values, trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ tf.transpose(game_embs_)

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix and (not self.p["Wfreeze"]):
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                for sl in self.ctx.slices:
                    print(f"slice = {sl}, score = {self.get_score(sl)}")
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [7]:
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [8]:
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

# Evals

In [9]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

def K_by_min(X):
    center = X.mean(axis=0)
    K = [euclidean_distances(X, center.reshape(1, -1)).argmax()]

    while len(K) < 100:
        K.append(euclidean_distances(X, X[K]).min(axis=1).argmax())
    return K


def eval_clustering(ctx, all_from_labels=False):
    X = ctx.get_relevs("train").T
    from sklearn.cluster import KMeans
    
    k_func = (
        (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
        if not all_from_labels else
        (lambda C : from_labels(X, C.labels_))
    )
    
    K_KMeans = k_func(
        KMeans(n_clusters=100, random_state=0).fit(X)  #, n_init="auto").fit(X)
    )


    from sklearn.cluster import BisectingKMeans
    K_BisectingKMeans = k_func(
        BisectingKMeans(n_clusters=100, random_state=0).fit(X)
    )


    from sklearn.cluster import MiniBatchKMeans
    K_MiniBatchKMeans = from_labels(
        X,
        MiniBatchKMeans(n_clusters=100, random_state=0).fit(X).labels_
    )

    from sklearn.cluster import AgglomerativeClustering
    K_AgglomerativeClustering = from_labels(
        X,
        AgglomerativeClustering(n_clusters=100).fit(X).labels_
    )

    from sklearn.cluster import SpectralCoclustering
    K_SpectralCoclustering = from_labels(
        X,
        SpectralCoclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralBiclustering
    K_SpectralBiclustering = from_labels(
        X,
        SpectralBiclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralClustering
    K_SpectralClusteringNN = from_labels(
        X,
        SpectralClustering(n_clusters=100, random_state=0, affinity="nearest_neighbors", n_jobs=-1).fit(X).labels_
    )

    K_ByMin = K_by_min(X)

    ev([
        Popular(ctx),
        AnnCUR(ctx),
        AnnCUR(ctx, key_games=np.arange(100), name="key_games=np.arange(100)"),
        AnnCUR(ctx, key_games=np.random.choice(np.arange(X.shape[0]), size=100, replace=False), name="random"),
        AnnCUR(ctx, key_games=K_KMeans, name="K_KMeans"),
        AnnCUR(ctx, key_games=K_BisectingKMeans, name="K_BisectingKMeans"),
        AnnCUR(ctx, key_games=K_MiniBatchKMeans, name="K_MiniBatchKMeans"),
        AnnCUR(ctx, key_games=K_AgglomerativeClustering, name="K_AgglomerativeClustering"),
        AnnCUR(ctx, key_games=K_SpectralCoclustering, name="K_SpectralCoclustering"),
        AnnCUR(ctx, key_games=K_SpectralBiclustering, name="K_SpectralBiclustering"),
        AnnCUR(ctx, key_games=K_SpectralClusteringNN, name="K_SpectralClusteringNN"),
        AnnCUR(ctx, key_games=K_ByMin, name="K_ByMin"),
    ])

In [10]:
DATASETS = ["yugioh", "pro_wrestling", "star_trek", "doctor_who", "military"]

# support items choice

In [11]:
# setup:
# 1000 items, 1000 + 1000 queries
recall_k = 5
n_runs = 10
n_items_per_run = 1000
test_item_cnt = 100
item_cnt_range = list(range(5, 101, 5))
assert test_item_cnt in item_cnt_range

In [12]:
from collections import defaultdict
from itertools import combinations

from matplotlib import pyplot as plt
import numpy as np
import os
from sklearn.metrics import top_k_accuracy_score
from sklearn.cluster import KMeans
from scipy.stats import mannwhitneyu, ttest_rel

In [13]:
#with open("collections_10000_items/test.npy", "rb") as fin:
#    all_test = np.load(fin)
#with open("collections_10000_items/train.npy", "rb") as fin:
#    all_train = np.load(fin)

#assert all_train.shape[0] == n_runs * n_items_per_run
#assert all_test.shape[0] == n_runs * n_items_per_run


In [14]:
def eval_items(items, train, test):
    item_embeds = train.dot(np.linalg.pinv(train[items]))
    approx_test_rel = item_embeds.dot(test[items])
    approx_train_rel = item_embeds.dot(train[items])
    best_items = np.argsort(test, axis=0)[-1]
    recall = top_k_accuracy_score(best_items, approx_test_rel.T, k=recall_k, labels=np.arange(train.shape[0]))
    test_mse = np.mean((test - approx_test_rel) ** 2)
    train_mse = np.mean((train - approx_train_rel) ** 2)
    return recall, test_mse, train_mse

def get_rnd_items(n_items, train):
    return np.random.choice(train.shape[0], n_items, replace=False)

def get_kmeans_items(n_items, train):
    kmeans = KMeans(n_clusters=n_items, n_init="auto").fit(train)
    items = []
    for center in kmeans.cluster_centers_:
        distances = ((train - center) ** 2).sum(axis=1)
        assert distances.shape == (train.shape[0],)
        for item_id in np.argsort(distances):
            if item_id not in items:
                items.append(item_id)
                break
    return np.array(items, dtype=np.int64)

greedy_items = []
def get_greedy(n_items, train, lazy=True):
    global greedy_items
    if not lazy:
        greedy_items = []
    if len(greedy_items) >= n_items:
        return np.array(greedy_items[:n_items], dtype=np.int64)
    
    gram = train.dot(train.T)
    gram /= gram.mean()
    items = []
    remaining_items = np.ones(train.shape[0], dtype="bool")
    for t in range(n_items):
        scores = (gram ** 2).sum(axis=1)
        assert np.allclose(scores[~remaining_items], np.zeros_like(scores[~remaining_items])), (
            t, scores[~remaining_items]
        )
        scores[remaining_items] /= gram[remaining_items, remaining_items]
        if max(scores) < 1e-9:
            break
        new_item = scores.argmax()
        assert remaining_items[new_item]
        coefs = gram[new_item] / np.sqrt(gram[new_item, new_item])
        diff = coefs.reshape(-1, 1).dot(coefs.reshape(1, -1))
        gram -= diff
        assert np.allclose(gram[new_item], np.zeros_like(gram[new_item]), atol=1e-6), (
            t, gram[new_item].std()
        )
        items.append(new_item)
        remaining_items[new_item] = False
    assert len(items) == n_items
    greedy_items.clear()
    greedy_items.extend(items)
    return np.array(items, dtype=np.int64)


### fast-a

In [15]:
def coitem_algorithm(n_support, candidate_items, target_items, min_item_rel_norm=1e-5, eps=1e-6):
    support_items = []
    support_items_scores = []
    n_c, n_q = candidate_items.shape
    n_t = target_items.shape[0]

    candidate_item_squared_norms = (candidate_items ** 2).sum(axis=1)
    min_item_norm = min_item_rel_norm * np.sqrt(candidate_item_squared_norms.mean())
    
    candidate_coitems = candidate_items.dot(
        target_items.T.dot(target_items)
    )
    orthonormed_support_items = np.zeros((n_support, n_q))
    remaining_items = np.ones(candidate_items.shape[0], dtype="bool")
    for t in tqdm.tqdm_notebook(range(n_support)):
        scores = (candidate_coitems * candidate_items).sum(axis=1)
        remaining_items &= (candidate_item_squared_norms >= min_item_norm ** 2)
        scores[remaining_items] /= candidate_item_squared_norms[remaining_items]
        scores[~remaining_items] = 0
        
        new_item_id = np.argmax(scores)
        assert remaining_items[new_item_id]
        support_items.append(new_item_id)
        support_items_scores.append(scores[new_item_id] / (n_t * n_q))
        remaining_items[new_item_id] = False
        
        new_item = candidate_items[new_item_id].copy()
        new_item -= orthonormed_support_items[:t].T.dot(
            orthonormed_support_items[:t].dot(new_item)
        )
        assert np.allclose((new_item ** 2).sum(), candidate_item_squared_norms[new_item_id], atol=eps)
        new_item /= np.sqrt((new_item ** 2).sum())
        orthonormed_support_items[t] = new_item
        new_coitem = candidate_coitems[new_item_id] / np.sqrt(candidate_item_squared_norms[new_item_id])
        
        coefs = (candidate_items * new_item).sum(axis=1)
        candidate_item_squared_norms -= coefs ** 2
        assert np.all(candidate_item_squared_norms[remaining_items] > -eps)
        
        candidate_coitems -= coefs.reshape((-1, 1)).dot(new_coitem.reshape((1, -1)))
        cocoefs = (candidate_coitems * new_item).sum(axis=1, keepdims=True)
        candidate_coitems -= cocoefs.dot(new_item.reshape((1, -1)))
    return np.array(support_items, dtype=np.int64), np.array(support_items_scores)

In [18]:
ctx.key_games

(array([5557, 6491, 1903,  248, 9446, 4744, 7244, 7104, 1121, 2338, 3864,
        5779, 7137, 2833,  381, 1230, 8071, 1332, 7252, 7008, 3836, 8421,
        2172, 6623, 6689, 8197, 4685, 9364, 5218, 8890, 9380, 1575, 2227,
        3020, 4791, 7807, 1805, 5783, 4066, 9838, 2355, 9200, 4128, 7461,
        5236, 1779, 3046, 5729, 2728, 6605, 9045, 2190, 7592, 1649, 6755,
          89, 6528, 8123, 7599, 2428,  957, 8020, 8998, 5831, 9227,  639,
        4832, 1214,  181, 4184, 4972, 1139, 5304, 6240, 3396, 6563, 3776,
        2262, 4805, 8566, 6095,  102, 9860, 3052, 5068, 3100, 5432, 1326,
        2501,  321,  186, 3516, 5841, 3959, 9276, 8843, 5563, 1502, 5900,
        6003]),
 array([0.34743124, 0.08071299, 0.04348799, 0.04027334, 0.02930198,
        0.0277006 , 0.02327564, 0.01926292, 0.01333025, 0.01233328,
        0.01029388, 0.00868232, 0.00840466, 0.00822039, 0.00820282,
        0.00737349, 0.00651845, 0.00609048, 0.00606077, 0.00505034,
        0.00490937, 0.00482551, 0.00459832, 0.

In [16]:
R = np.load('./R1000train.npz')
R["R"].shape

(1000, 8223)

In [20]:
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=100
)

Best det =  0.0
Current de =  0.0
100 599 300


In [25]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)





model =  <__main__.Popular object at 0x7fb6e624cc50>
np.mean(results), mse, len(results) =  0.025575959933222037 0.0054122903 599
np.mean(results), mse, len(results) =  0.021199999999999997 0.0055082804 300
0.025575959933222037 0.021199999999999997



model =  AnnCur(140423420926096)
np.mean(results), mse, len(results) =  0.6999833055091819 0.0011132385 599
np.mean(results), mse, len(results) =  0.6327333333333333 0.0016203691 300
0.6999833055091819 0.6327333333333333



model =  AnnCur(key_games=np.arange(100) - 140423419826128)
np.mean(results), mse, len(results) =  0.6657929883138565 0.0013816745 599
np.mean(results), mse, len(results) =  0.5838000000000001 0.0020310022 300
0.6657929883138565 0.5838000000000001



model =  AnnCur(random - 140423506349776)
np.mean(results), mse, len(results) =  0.6997829716193656 0.0011298328 599
np.mean(results), mse, len(results) =  0.6371333333333333 0.0016117776 300
0.6997829716193656 0.6371333333333333



model =  AnnCur(K_KMeans - 1404234216

In [21]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/var/tmp/ipykernel_326286/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.7360267111853088 0.0008644789 599
np.mean(results), mse, len(results) =  0.6599666666666667 0.0012965548 300


(0.7360267111853088, 0.6599666666666667)

In [18]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.001
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (599, 100)
self.embed_games.shape =  (8223, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8223)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (599, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.3329000000000001 tf.Tensor(0.008599035716251481, shape=(), dtype=float64) 100
slice = key, score = 0.3329000000000001
np.mean(results), mse, len(results) =  0.37405676126878135 tf.Tensor(0.008170502513637568, shape=(), dtype=float64) 599
slice = train, score = 0.37405676126878135
np.mean(results), mse, len(results) =  0.34296666666666664 tf.Tensor(0.008794909448954091, shape=(), dtype=float64) 300
slice = test, score = 0.34296666666666664

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.431 tf.Tensor(89.61607972499597, shape=(), dtype=float64) 100
slice = key, score = 0.431
np.mean(results), mse, len(results) =  0.4347746243739566 tf.Tensor(83.27372232628372, sh

KeyboardInterrupt: 

In [25]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': False,
    'train_vbias': True,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'mse',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (599, 100)
self.embed_games.shape =  (8223, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8223)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (599, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.6704000000000001 tf.Tensor(0.0013085468222526315, shape=(), dtype=float64) 100
slice = key, score = 0.6704000000000001
np.mean(results), mse, len(results) =  0.7360601001669449 tf.Tensor(0.0008644753498525569, shape=(), dtype=float64) 599
slice = train, score = 0.7360601001669449
np.mean(results), mse, len(results) =  0.6599333333333334 tf.Tensor(0.001296546580314037, shape=(), dtype=float64) 300
slice = test, score = 0.6599333333333334

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.6705000000000001 tf.Tensor(0.0013080316202927387, shape=(), dtype=float64) 100
slice = key, score = 0.6705000000000001
np.mean(results), mse, len(results) =  0.7351585976627714 tf.

np.mean(results), mse, len(results) =  0.6599333333333334 tf.Tensor(0.001299111864223921, shape=(), dtype=float64) 300
slice = test, score = 0.6599333333333334

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.6707 tf.Tensor(0.0013076398381858432, shape=(), dtype=float64) 100
slice = key, score = 0.6707
np.mean(results), mse, len(results) =  0.7356260434056761 tf.Tensor(0.0008670399965183635, shape=(), dtype=float64) 599
slice = train, score = 0.7356260434056761
np.mean(results), mse, len(results) =  0.6593333333333333 tf.Tensor(0.001298896595390865, shape=(), dtype=float64) 300
slice = test, score = 0.6593333333333333


KeyboardInterrupt: 

In [26]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': False,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'mse',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (599, 100)
self.embed_games.shape =  (8223, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8223)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (599, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0079 tf.Tensor(0.011844911, shape=(), dtype=float32) 100
slice = key, score = 0.0079
np.mean(results), mse, len(results) =  0.013055091819699499 tf.Tensor(0.012485181, shape=(), dtype=float32) 599
slice = train, score = 0.013055091819699499
np.mean(results), mse, len(results) =  0.013166666666666667 tf.Tensor(0.012034644, shape=(), dtype=float32) 300
slice = test, score = 0.013166666666666667

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.6396000000000001 tf.Tensor(0.0014336883, shape=(), dtype=float32) 100
slice = key, score = 0.6396000000000001
np.mean(results), mse, len(results) =  0.6795325542570951 tf.Tensor(0.0011860357, shape=(), dtype=float32) 599
slic


=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.6973999999999999 tf.Tensor(0.0011005008, shape=(), dtype=float32) 100
slice = key, score = 0.6973999999999999
np.mean(results), mse, len(results) =  0.7464273789649415 tf.Tensor(0.00080342806, shape=(), dtype=float32) 599
slice = train, score = 0.7464273789649415
np.mean(results), mse, len(results) =  0.65 tf.Tensor(0.0013723474, shape=(), dtype=float32) 300
slice = test, score = 0.65


KeyboardInterrupt: 

In [27]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': False,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (599, 100)
self.embed_games.shape =  (8223, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8223)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (599, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.015100000000000002 tf.Tensor(0.013193638, shape=(), dtype=float32) 100
slice = key, score = 0.015100000000000002
np.mean(results), mse, len(results) =  0.014590984974958265 tf.Tensor(0.013282064, shape=(), dtype=float32) 599
slice = train, score = 0.014590984974958265
np.mean(results), mse, len(results) =  0.013000000000000001 tf.Tensor(0.012977752, shape=(), dtype=float32) 300
slice = test, score = 0.013000000000000001

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.29949999999999993 tf.Tensor(39.121723, shape=(), dtype=float32) 100
slice = key, score = 0.29949999999999993
np.mean(results), mse, len(results) =  0.31225375626043406 tf.Tensor(41.619648, shape=()


=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.6166 tf.Tensor(346.97757, shape=(), dtype=float32) 100
slice = key, score = 0.6166
np.mean(results), mse, len(results) =  0.6352253756260434 tf.Tensor(346.78094, shape=(), dtype=float32) 599
slice = train, score = 0.6352253756260434
np.mean(results), mse, len(results) =  0.5095333333333333 tf.Tensor(317.70096, shape=(), dtype=float32) 300
slice = test, score = 0.5095333333333333

=== Iteration 19000 ===
np.mean(results), mse, len(results) =  0.6187000000000001 tf.Tensor(356.87405, shape=(), dtype=float32) 100
slice = key, score = 0.6187000000000001
np.mean(results), mse, len(results) =  0.6397495826377295 tf.Tensor(356.72064, shape=(), dtype=float32) 599
slice = train, score = 0.6397495826377295
np.mean(results), mse, len(results) =  0.5155666666666666 tf.Tensor(328.53674, shape=(), dtype=float32) 300
slice = test, score = 0.5155666666666666

=== Iteration 20000 ===
np.mean(results), mse, len(results) =  0.622 tf.Tensor(

KeyboardInterrupt: 

In [28]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'mse',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (599, 100)
self.embed_games.shape =  (8223, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8223)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (599, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.37059999999999993 tf.Tensor(0.007621676759208236, shape=(), dtype=float64) 100
slice = key, score = 0.37059999999999993
np.mean(results), mse, len(results) =  0.3972787979966611 tf.Tensor(0.007950330600329696, shape=(), dtype=float64) 599
slice = train, score = 0.3972787979966611
np.mean(results), mse, len(results) =  0.36136666666666667 tf.Tensor(0.00790850133890362, shape=(), dtype=float64) 300
slice = test, score = 0.36136666666666667

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.6762000000000001 tf.Tensor(0.0012336373137285112, shape=(), dtype=float64) 100
slice = key, score = 0.6762000000000001
np.mean(results), mse, len(results) =  0.7336727879799667 tf

KeyboardInterrupt: 

In [29]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'mse',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (599, 100)
self.embed_games.shape =  (8223, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8223)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (599, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.3556 tf.Tensor(0.0075319774625899494, shape=(), dtype=float64) 100
slice = key, score = 0.3556
np.mean(results), mse, len(results) =  0.39901502504173625 tf.Tensor(0.00756115258105171, shape=(), dtype=float64) 599
slice = train, score = 0.39901502504173625
np.mean(results), mse, len(results) =  0.3638666666666667 tf.Tensor(0.007621635033128592, shape=(), dtype=float64) 300
slice = test, score = 0.3638666666666667

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.6676000000000001 tf.Tensor(0.0013147807806401335, shape=(), dtype=float64) 100
slice = key, score = 0.6676000000000001
np.mean(results), mse, len(results) =  0.7356427378964941 tf.Tensor(0.000868085035818

np.mean(results), mse, len(results) =  0.753272120200334 tf.Tensor(0.0007584964107236591, shape=(), dtype=float64) 599
slice = train, score = 0.753272120200334
np.mean(results), mse, len(results) =  0.6478666666666666 tf.Tensor(0.0014175884974529178, shape=(), dtype=float64) 300
slice = test, score = 0.6478666666666666

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.6607000000000001 tf.Tensor(0.0014214904289952964, shape=(), dtype=float64) 100
slice = key, score = 0.6607000000000001
np.mean(results), mse, len(results) =  0.7542904841402337 tf.Tensor(0.000750484499375416, shape=(), dtype=float64) 599
slice = train, score = 0.7542904841402337
np.mean(results), mse, len(results) =  0.6473 tf.Tensor(0.001424100624033018, shape=(), dtype=float64) 300
slice = test, score = 0.6473

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.6597 tf.Tensor(0.0014292752759138647, shape=(), dtype=float64) 100
slice = key, score = 0.6597
np.mean(results), mse, len(results) 

np.mean(results), mse, len(results) =  0.6399000000000001 tf.Tensor(0.001511943741090673, shape=(), dtype=float64) 300
slice = test, score = 0.6399000000000001

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.6515 tf.Tensor(0.0015003713359317123, shape=(), dtype=float64) 100
slice = key, score = 0.6515
np.mean(results), mse, len(results) =  0.7619031719532554 tf.Tensor(0.0007124800150251809, shape=(), dtype=float64) 599
slice = train, score = 0.7619031719532554
np.mean(results), mse, len(results) =  0.6396 tf.Tensor(0.0015126232705544391, shape=(), dtype=float64) 300
slice = test, score = 0.6396

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.6501 tf.Tensor(0.0014967752808317367, shape=(), dtype=float64) 100
slice = key, score = 0.6501
np.mean(results), mse, len(results) =  0.7625542570951586 tf.Tensor(0.0007092707683464259, shape=(), dtype=float64) 599
slice = train, score = 0.7625542570951586
np.mean(results), mse, len(results) =  0.6407333333333334

np.mean(results), mse, len(results) =  0.6341666666666667 tf.Tensor(0.0015904042130781052, shape=(), dtype=float64) 300
slice = test, score = 0.6341666666666667

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.6446999999999999 tf.Tensor(0.001588736377822469, shape=(), dtype=float64) 100
slice = key, score = 0.6446999999999999
np.mean(results), mse, len(results) =  0.7687145242070117 tf.Tensor(0.0006903591092789119, shape=(), dtype=float64) 599
slice = train, score = 0.7687145242070117
np.mean(results), mse, len(results) =  0.6325999999999999 tf.Tensor(0.0016051984160958747, shape=(), dtype=float64) 300
slice = test, score = 0.6325999999999999

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.6428 tf.Tensor(0.0015904054673823843, shape=(), dtype=float64) 100
slice = key, score = 0.6428
np.mean(results), mse, len(results) =  0.7692320534223707 tf.Tensor(0.0006803552286472921, shape=(), dtype=float64) 599
slice = train, score = 0.7692320534223707
np.mean(r

np.mean(results), mse, len(results) =  0.6281 tf.Tensor(0.0016775724300267829, shape=(), dtype=float64) 300
slice = test, score = 0.6281

=== Iteration 68000 ===
np.mean(results), mse, len(results) =  0.6364 tf.Tensor(0.0016546087809218304, shape=(), dtype=float64) 100
slice = key, score = 0.6364
np.mean(results), mse, len(results) =  0.7729549248747913 tf.Tensor(0.000666245209729196, shape=(), dtype=float64) 599
slice = train, score = 0.7729549248747913
np.mean(results), mse, len(results) =  0.6285333333333334 tf.Tensor(0.0016752794432300091, shape=(), dtype=float64) 300
slice = test, score = 0.6285333333333334

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.6373 tf.Tensor(0.00166990722881152, shape=(), dtype=float64) 100
slice = key, score = 0.6373
np.mean(results), mse, len(results) =  0.7731051752921536 tf.Tensor(0.0006658600814935436, shape=(), dtype=float64) 599
slice = train, score = 0.7731051752921536
np.mean(results), mse, len(results) =  0.6278333333333334 t

np.mean(results), mse, len(results) =  0.6239666666666666 tf.Tensor(0.0017427419797337858, shape=(), dtype=float64) 300
slice = test, score = 0.6239666666666666

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.6350999999999999 tf.Tensor(0.0017099701342068831, shape=(), dtype=float64) 100
slice = key, score = 0.6350999999999999
np.mean(results), mse, len(results) =  0.7747245409015026 tf.Tensor(0.0006603897053333365, shape=(), dtype=float64) 599
slice = train, score = 0.7747245409015026
np.mean(results), mse, len(results) =  0.6252666666666666 tf.Tensor(0.001731925119858559, shape=(), dtype=float64) 300
slice = test, score = 0.6252666666666666

=== Iteration 86000 ===
np.mean(results), mse, len(results) =  0.6310000000000001 tf.Tensor(0.0017183394312403008, shape=(), dtype=float64) 100
slice = key, score = 0.6310000000000001
np.mean(results), mse, len(results) =  0.7756928213689482 tf.Tensor(0.0006544939216605017, shape=(), dtype=float64) 599
slice = train, score = 0.77

(0.7768113522537563, 0.6195333333333334)

In [30]:
!ls

am_download_data
doctor_who
log.local.bin.old
log.local.logtime2.bin
log.local.logtime2.true.bin
log.local.txt.old.bin
matrix_approx_zeshel.py
military
proof-of-concept-open-data-round5-5-parallel.ipynb
proof-of-concept-open-data-round5-new-key.ipynb
proof-of-concept-open-data-round6-greedykey.ipynb
proof-of-concept-open-data-round6-greedykey-profile.ipynb
proof-of-concept-open-data-round7-ms-query.ipynb
pro_wrestling
__pycache__
R1000train.npz
R2000train.npz
stand
star_trek
test_support_item_choice_strategies_several_subamples-open_data.ipynb
yugioh


==============

In [31]:
!ls

am_download_data
doctor_who
log.local.bin.old
log.local.logtime2.bin
log.local.logtime2.true.bin
log.local.txt.old.bin
matrix_approx_zeshel.py
military
proof-of-concept-open-data-round5-5-parallel.ipynb
proof-of-concept-open-data-round5-new-key.ipynb
proof-of-concept-open-data-round6-greedykey.ipynb
proof-of-concept-open-data-round6-greedykey-profile.ipynb
proof-of-concept-open-data-round7-ms-query.ipynb
pro_wrestling
__pycache__
R1000train.npz
R2000train.npz
R3000test.npz
R3000train.npz
R5000test.npz
stand
star_trek
test_support_item_choice_strategies_several_subamples-open_data.ipynb
yugioh


In [34]:
R = np.load('./R3000test.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 1999 900


In [35]:
eval_clustering(ctx, all_from_labels=True)

ImportError: cannot import name 'BisectingKMeans' from 'sklearn.cluster' (/home/shevkunov/anaconda3/lib/python3.9/site-packages/sklearn/cluster/__init__.py)

In [36]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/var/tmp/ipykernel_326286/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6689094547273637 0.000980972 1999
np.mean(results), mse, len(results) =  0.6458777777777778 0.0011021171 900


(0.6689094547273637, 0.6458777777777778)

In [37]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': False, 'train_bias': True,
    'verbose': False, 'loss': 'mse',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1999, 100)
self.embed_games.shape =  (24493, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 24493)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1999, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.2907 tf.Tensor(0.007907760550851186, shape=(), dtype=float64) 100
slice = key, score = 0.2907
np.mean(results), mse, len(results) =  0.33713356678339174 tf.Tensor(0.007317207780828992, shape=(), dtype=float64) 1999
slice = train, score = 0.33713356678339174
np.mean(results), mse, len(results) =  0.3233333333333333 tf.Tensor(0.007510761179203233, shape=(), dtype=float64) 900
slice = test, score = 0.3233333333333333

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.6154000000000001 tf.Tensor(0.0011351383589865284, shape=(), dtype=float64) 100
slice = key, score = 0.6154000000000001
np.mean(results), mse, len(results) =  0.6665082541270635 tf.Tensor(0.0009952700

np.mean(results), mse, len(results) =  0.6744422211105553 tf.Tensor(0.0009437971101682594, shape=(), dtype=float64) 1999
slice = train, score = 0.6744422211105553
np.mean(results), mse, len(results) =  0.6468444444444444 tf.Tensor(0.001106158975578018, shape=(), dtype=float64) 900
slice = test, score = 0.6468444444444444

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.6187 tf.Tensor(0.001127937681558827, shape=(), dtype=float64) 100
slice = key, score = 0.6187
np.mean(results), mse, len(results) =  0.6750175087543772 tf.Tensor(0.0009428431684069343, shape=(), dtype=float64) 1999
slice = train, score = 0.6750175087543772
np.mean(results), mse, len(results) =  0.6473555555555555 tf.Tensor(0.0011071778359131364, shape=(), dtype=float64) 900
slice = test, score = 0.6473555555555555

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.6208 tf.Tensor(0.0011259974264312218, shape=(), dtype=float64) 100
slice = key, score = 0.6208
np.mean(results), mse, len(resul

np.mean(results), mse, len(results) =  0.678839419709855 tf.Tensor(0.0009238403481145037, shape=(), dtype=float64) 1999
slice = train, score = 0.678839419709855
np.mean(results), mse, len(results) =  0.646611111111111 tf.Tensor(0.0011082998631763872, shape=(), dtype=float64) 900
slice = test, score = 0.646611111111111

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.6194000000000001 tf.Tensor(0.0011269976454026348, shape=(), dtype=float64) 100
slice = key, score = 0.6194000000000001
np.mean(results), mse, len(results) =  0.6782191095547774 tf.Tensor(0.0009231267967512688, shape=(), dtype=float64) 1999
slice = train, score = 0.6782191095547774
np.mean(results), mse, len(results) =  0.6461555555555555 tf.Tensor(0.0011096947101945046, shape=(), dtype=float64) 900
slice = test, score = 0.6461555555555555

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.6225999999999999 tf.Tensor(0.0011243386713851193, shape=(), dtype=float64) 100
slice = key, score = 0.622

np.mean(results), mse, len(results) =  0.6832166083041521 tf.Tensor(0.0009065299632260332, shape=(), dtype=float64) 1999
slice = train, score = 0.6832166083041521
np.mean(results), mse, len(results) =  0.6492777777777777 tf.Tensor(0.0011089009264846623, shape=(), dtype=float64) 900
slice = test, score = 0.6492777777777777

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.6226999999999999 tf.Tensor(0.0011298134047603764, shape=(), dtype=float64) 100
slice = key, score = 0.6226999999999999
np.mean(results), mse, len(results) =  0.683271635817909 tf.Tensor(0.0009069851727376039, shape=(), dtype=float64) 1999
slice = train, score = 0.683271635817909
np.mean(results), mse, len(results) =  0.6485888888888889 tf.Tensor(0.0011093589358177364, shape=(), dtype=float64) 900
slice = test, score = 0.6485888888888889

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.6223000000000001 tf.Tensor(0.0011268890954078134, shape=(), dtype=float64) 100
slice = key, score = 0.6


=== Iteration 67000 ===
np.mean(results), mse, len(results) =  0.6212000000000001 tf.Tensor(0.0011312507108843892, shape=(), dtype=float64) 100
slice = key, score = 0.6212000000000001
np.mean(results), mse, len(results) =  0.685247623811906 tf.Tensor(0.0008936082678168917, shape=(), dtype=float64) 1999
slice = train, score = 0.685247623811906
np.mean(results), mse, len(results) =  0.6487 tf.Tensor(0.0011097956962417364, shape=(), dtype=float64) 900
slice = test, score = 0.6487

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.6221 tf.Tensor(0.0011299566443456548, shape=(), dtype=float64) 100
slice = key, score = 0.6221
np.mean(results), mse, len(results) =  0.686168084042021 tf.Tensor(0.0008933515943865847, shape=(), dtype=float64) 1999
slice = train, score = 0.686168084042021
np.mean(results), mse, len(results) =  0.6490222222222223 tf.Tensor(0.0011101367394287095, shape=(), dtype=float64) 900
slice = test, score = 0.6490222222222223

=== Iteration 70000 ===
np.mean(r

np.mean(results), mse, len(results) =  0.6499666666666667 tf.Tensor(0.0011111455872031518, shape=(), dtype=float64) 900
slice = test, score = 0.6499666666666667

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.6238 tf.Tensor(0.0011347761500350552, shape=(), dtype=float64) 100
slice = key, score = 0.6238
np.mean(results), mse, len(results) =  0.686863431715858 tf.Tensor(0.0008883466460459467, shape=(), dtype=float64) 1999
slice = train, score = 0.686863431715858
np.mean(results), mse, len(results) =  0.6488666666666667 tf.Tensor(0.0011130166620734268, shape=(), dtype=float64) 900
slice = test, score = 0.6488666666666667

=== Iteration 86000 ===
np.mean(results), mse, len(results) =  0.6254000000000001 tf.Tensor(0.00113574160435351, shape=(), dtype=float64) 100
slice = key, score = 0.6254000000000001
np.mean(results), mse, len(results) =  0.6881990995497749 tf.Tensor(0.0008867296573078073, shape=(), dtype=float64) 1999
slice = train, score = 0.6881990995497749
np.mean(re

(0.6882491245622812, 0.6489555555555556)

In [38]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': False, 'train_bias': True,
    'verbose': False, 'loss': 'mse',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": False,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1999, 100)
self.embed_games.shape =  (24493, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 24493)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1999, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.32010000000000005 tf.Tensor(0.0073608266309495955, shape=(), dtype=float64) 100
slice = key, score = 0.32010000000000005
np.mean(results), mse, len(results) =  0.341295647823912 tf.Tensor(0.007283653789119284, shape=(), dtype=float64) 1999
slice = train, score = 0.341295647823912
np.mean(results), mse, len(results) =  0.33161111111111113 tf.Tensor(0.007654977221320613, shape=(), dtype=float64) 900
slice = test, score = 0.33161111111111113

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.6165 tf.Tensor(0.0011313937291074167, shape=(), dtype=float64) 100
slice = key, score = 0.6165
np.mean(results), mse, len(results) =  0.6662931465732866 tf.Tensor(0.000992573

np.mean(results), mse, len(results) =  0.6766533266633317 tf.Tensor(0.0009368585977165042, shape=(), dtype=float64) 1999
slice = train, score = 0.6766533266633317
np.mean(results), mse, len(results) =  0.6475333333333333 tf.Tensor(0.001103645246335358, shape=(), dtype=float64) 900
slice = test, score = 0.6475333333333333

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.6208 tf.Tensor(0.0011229368370448234, shape=(), dtype=float64) 100
slice = key, score = 0.6208
np.mean(results), mse, len(results) =  0.6763231615807904 tf.Tensor(0.000937161074297256, shape=(), dtype=float64) 1999
slice = train, score = 0.6763231615807904
np.mean(results), mse, len(results) =  0.6465111111111111 tf.Tensor(0.0011061714581556627, shape=(), dtype=float64) 900
slice = test, score = 0.6465111111111111

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.6204999999999999 tf.Tensor(0.0011237570737152435, shape=(), dtype=float64) 100
slice = key, score = 0.6204999999999999
np.mean(

np.mean(results), mse, len(results) =  0.6790345172586293 tf.Tensor(0.0009183614284418714, shape=(), dtype=float64) 1999
slice = train, score = 0.6790345172586293
np.mean(results), mse, len(results) =  0.6469111111111111 tf.Tensor(0.0011078437090790158, shape=(), dtype=float64) 900
slice = test, score = 0.6469111111111111

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.6206 tf.Tensor(0.0011281732001317045, shape=(), dtype=float64) 100
slice = key, score = 0.6206
np.mean(results), mse, len(results) =  0.679559779889945 tf.Tensor(0.0009173174728985949, shape=(), dtype=float64) 1999
slice = train, score = 0.679559779889945
np.mean(results), mse, len(results) =  0.6471555555555556 tf.Tensor(0.0011088514245655087, shape=(), dtype=float64) 900
slice = test, score = 0.6471555555555556

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.6202000000000001 tf.Tensor(0.0011289026691448745, shape=(), dtype=float64) 100
slice = key, score = 0.6202000000000001
np.mean(

np.mean(results), mse, len(results) =  0.6817208604302151 tf.Tensor(0.0009070648518971987, shape=(), dtype=float64) 1999
slice = train, score = 0.6817208604302151
np.mean(results), mse, len(results) =  0.6465555555555556 tf.Tensor(0.0011114337647880762, shape=(), dtype=float64) 900
slice = test, score = 0.6465555555555556

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.6185 tf.Tensor(0.0011362899648516237, shape=(), dtype=float64) 100
slice = key, score = 0.6185
np.mean(results), mse, len(results) =  0.6828714357178589 tf.Tensor(0.0009093536902205077, shape=(), dtype=float64) 1999
slice = train, score = 0.6828714357178589
np.mean(results), mse, len(results) =  0.6483777777777777 tf.Tensor(0.0011128486941574343, shape=(), dtype=float64) 900
slice = test, score = 0.6483777777777777

=== Iteration 53000 ===
np.mean(results), mse, len(results) =  0.6187999999999999 tf.Tensor(0.0011343583708485424, shape=(), dtype=float64) 100
slice = key, score = 0.6187999999999999
np.mea

np.mean(results), mse, len(results) =  0.6838269134567285 tf.Tensor(0.0009020480151353247, shape=(), dtype=float64) 1999
slice = train, score = 0.6838269134567285
np.mean(results), mse, len(results) =  0.6474444444444445 tf.Tensor(0.0011159288140538102, shape=(), dtype=float64) 900
slice = test, score = 0.6474444444444445

=== Iteration 70000 ===
np.mean(results), mse, len(results) =  0.6193 tf.Tensor(0.001136099735575894, shape=(), dtype=float64) 100
slice = key, score = 0.6193
np.mean(results), mse, len(results) =  0.684072036018009 tf.Tensor(0.0009004914610386052, shape=(), dtype=float64) 1999
slice = train, score = 0.684072036018009
np.mean(results), mse, len(results) =  0.6475555555555556 tf.Tensor(0.0011149411558797305, shape=(), dtype=float64) 900
slice = test, score = 0.6475555555555556

=== Iteration 71000 ===
np.mean(results), mse, len(results) =  0.6186 tf.Tensor(0.0011377679924042588, shape=(), dtype=float64) 100
slice = key, score = 0.6186
np.mean(results), mse, len(result


=== Iteration 87000 ===
np.mean(results), mse, len(results) =  0.6182000000000001 tf.Tensor(0.001138837771915132, shape=(), dtype=float64) 100
slice = key, score = 0.6182000000000001
np.mean(results), mse, len(results) =  0.6851375687843922 tf.Tensor(0.0008948878120538803, shape=(), dtype=float64) 1999
slice = train, score = 0.6851375687843922
np.mean(results), mse, len(results) =  0.6478111111111111 tf.Tensor(0.0011175853677122252, shape=(), dtype=float64) 900
slice = test, score = 0.6478111111111111

=== Iteration 88000 ===
np.mean(results), mse, len(results) =  0.6179000000000001 tf.Tensor(0.0011409789864578546, shape=(), dtype=float64) 100
slice = key, score = 0.6179000000000001
np.mean(results), mse, len(results) =  0.6846823411705855 tf.Tensor(0.0008954859256702226, shape=(), dtype=float64) 1999
slice = train, score = 0.6846823411705855
np.mean(results), mse, len(results) =  0.6461444444444444 tf.Tensor(0.0011196479278122822, shape=(), dtype=float64) 900
slice = test, score = 0.

(0.6857028514257129, 0.6471222222222222)

In [39]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': False, 'train_bias': True,
    'verbose': False, 'loss': 'softmax',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": False,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1999, 100)
self.embed_games.shape =  (24493, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 24493)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1999, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.3428999999999999 tf.Tensor(0.006794424860178713, shape=(), dtype=float64) 100
slice = key, score = 0.3428999999999999
np.mean(results), mse, len(results) =  0.3590695347673837 tf.Tensor(0.006904476766363904, shape=(), dtype=float64) 1999
slice = train, score = 0.3590695347673837
np.mean(results), mse, len(results) =  0.3501888888888889 tf.Tensor(0.007022737246161761, shape=(), dtype=float64) 900
slice = test, score = 0.3501888888888889

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.1757 tf.Tensor(41.20711298999198, shape=(), dtype=float64) 100
slice = key, score = 0.1757
np.mean(results), mse, len(results) =  0.2173336668334167 tf.Tensor(45.29601379569538,

np.mean(results), mse, len(results) =  0.5211111111111111 tf.Tensor(277.15694086224477, shape=(), dtype=float64) 900
slice = test, score = 0.5211111111111111

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.525 tf.Tensor(265.6526421665262, shape=(), dtype=float64) 100
slice = key, score = 0.525
np.mean(results), mse, len(results) =  0.56991995997999 tf.Tensor(273.1804001475212, shape=(), dtype=float64) 1999
slice = train, score = 0.56991995997999
np.mean(results), mse, len(results) =  0.5262111111111112 tf.Tensor(278.06845315579926, shape=(), dtype=float64) 900
slice = test, score = 0.5262111111111112

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.5261 tf.Tensor(264.53284643795615, shape=(), dtype=float64) 100
slice = key, score = 0.5261
np.mean(results), mse, len(results) =  0.5751875937968983 tf.Tensor(276.44659355868504, shape=(), dtype=float64) 1999
slice = train, score = 0.5751875937968983
np.mean(results), mse, len(results) =  0.530244444444444


=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.5527000000000001 tf.Tensor(367.60835823105265, shape=(), dtype=float64) 100
slice = key, score = 0.5527000000000001
np.mean(results), mse, len(results) =  0.6163481740870436 tf.Tensor(362.2480494549728, shape=(), dtype=float64) 1999
slice = train, score = 0.6163481740870436
np.mean(results), mse, len(results) =  0.5718 tf.Tensor(367.73441722141115, shape=(), dtype=float64) 900
slice = test, score = 0.5718

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.5562 tf.Tensor(368.1881571144068, shape=(), dtype=float64) 100
slice = key, score = 0.5562
np.mean(results), mse, len(results) =  0.6176238119059531 tf.Tensor(367.50868570347825, shape=(), dtype=float64) 1999
slice = train, score = 0.6176238119059531
np.mean(results), mse, len(results) =  0.5730333333333334 tf.Tensor(371.99253887126986, shape=(), dtype=float64) 900
slice = test, score = 0.5730333333333334

=== Iteration 36000 ===
np.mean(results), mse, le

np.mean(results), mse, len(results) =  0.636928464232116 tf.Tensor(426.3441386151139, shape=(), dtype=float64) 1999
slice = train, score = 0.636928464232116
np.mean(results), mse, len(results) =  0.5897999999999999 tf.Tensor(432.61742104169605, shape=(), dtype=float64) 900
slice = test, score = 0.5897999999999999

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.5718999999999999 tf.Tensor(431.87286669079305, shape=(), dtype=float64) 100
slice = key, score = 0.5718999999999999
np.mean(results), mse, len(results) =  0.6379339669834918 tf.Tensor(427.7520904672694, shape=(), dtype=float64) 1999
slice = train, score = 0.6379339669834918
np.mean(results), mse, len(results) =  0.5903333333333333 tf.Tensor(433.89629936001353, shape=(), dtype=float64) 900
slice = test, score = 0.5903333333333333

=== Iteration 53000 ===
np.mean(results), mse, len(results) =  0.5708999999999999 tf.Tensor(434.94745702157303, shape=(), dtype=float64) 100
slice = key, score = 0.5708999999999999
np.m

np.mean(results), mse, len(results) =  0.6516758379189596 tf.Tensor(462.4388598543195, shape=(), dtype=float64) 1999
slice = train, score = 0.6516758379189596
np.mean(results), mse, len(results) =  0.6042 tf.Tensor(469.3295917793677, shape=(), dtype=float64) 900
slice = test, score = 0.6042

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.5882000000000001 tf.Tensor(486.62073338035646, shape=(), dtype=float64) 100
slice = key, score = 0.5882000000000001
np.mean(results), mse, len(results) =  0.6524662331165583 tf.Tensor(458.7451378905471, shape=(), dtype=float64) 1999
slice = train, score = 0.6524662331165583
np.mean(results), mse, len(results) =  0.6031333333333334 tf.Tensor(465.2800542024613, shape=(), dtype=float64) 900
slice = test, score = 0.6031333333333334

=== Iteration 70000 ===
np.mean(results), mse, len(results) =  0.5889 tf.Tensor(500.4523356189431, shape=(), dtype=float64) 100
slice = key, score = 0.5889
np.mean(results), mse, len(results) =  0.652996498249

np.mean(results), mse, len(results) =  0.6627113556778389 tf.Tensor(500.06856978022506, shape=(), dtype=float64) 1999
slice = train, score = 0.6627113556778389
np.mean(results), mse, len(results) =  0.6132777777777777 tf.Tensor(509.4068562081868, shape=(), dtype=float64) 900
slice = test, score = 0.6132777777777777

=== Iteration 86000 ===
np.mean(results), mse, len(results) =  0.5972999999999999 tf.Tensor(538.686778284996, shape=(), dtype=float64) 100
slice = key, score = 0.5972999999999999
np.mean(results), mse, len(results) =  0.6632516258129065 tf.Tensor(503.7438721378788, shape=(), dtype=float64) 1999
slice = train, score = 0.6632516258129065
np.mean(results), mse, len(results) =  0.6148888888888888 tf.Tensor(512.3927651392149, shape=(), dtype=float64) 900
slice = test, score = 0.6148888888888888

=== Iteration 87000 ===
np.mean(results), mse, len(results) =  0.5965 tf.Tensor(545.6308142920547, shape=(), dtype=float64) 100
slice = key, score = 0.5965
np.mean(results), mse, len(res

(0.6704102051025513, 0.6194444444444445)

In [40]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1999, 100)
self.embed_games.shape =  (24493, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 24493)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1999, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0076 tf.Tensor(37.992577, shape=(), dtype=float32) 100
slice = key, score = 0.0076
np.mean(results), mse, len(results) =  0.005427713856928464 tf.Tensor(39.957115, shape=(), dtype=float32) 1999
slice = train, score = 0.005427713856928464
np.mean(results), mse, len(results) =  0.005055555555555555 tf.Tensor(40.072792, shape=(), dtype=float32) 900
slice = test, score = 0.005055555555555555

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.565 tf.Tensor(47.54262, shape=(), dtype=float32) 100
slice = key, score = 0.565
np.mean(results), mse, len(results) =  0.6378989494747372 tf.Tensor(47.944195, shape=(), dtype=float32) 1999
slice = train, score = 0.637898949474

np.mean(results), mse, len(results) =  0.6542 tf.Tensor(295.1569, shape=(), dtype=float32) 900
slice = test, score = 0.6542

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.6164999999999999 tf.Tensor(299.47998, shape=(), dtype=float32) 100
slice = key, score = 0.6164999999999999
np.mean(results), mse, len(results) =  0.7113056528264132 tf.Tensor(300.52545, shape=(), dtype=float32) 1999
slice = train, score = 0.7113056528264132
np.mean(results), mse, len(results) =  0.6561888888888889 tf.Tensor(300.65585, shape=(), dtype=float32) 900
slice = test, score = 0.6561888888888889

=== Iteration 19000 ===
np.mean(results), mse, len(results) =  0.6211 tf.Tensor(306.41782, shape=(), dtype=float32) 100
slice = key, score = 0.6211
np.mean(results), mse, len(results) =  0.7121810905452727 tf.Tensor(308.77936, shape=(), dtype=float32) 1999
slice = train, score = 0.7121810905452727
np.mean(results), mse, len(results) =  0.6565666666666666 tf.Tensor(308.67426, shape=(), dtype=float32)

np.mean(results), mse, len(results) =  0.6565111111111112 tf.Tensor(417.06833, shape=(), dtype=float32) 900
slice = test, score = 0.6565111111111112

=== Iteration 36000 ===
np.mean(results), mse, len(results) =  0.6175 tf.Tensor(413.06638, shape=(), dtype=float32) 100
slice = key, score = 0.6175
np.mean(results), mse, len(results) =  0.721495747873937 tf.Tensor(415.45743, shape=(), dtype=float32) 1999
slice = train, score = 0.721495747873937
np.mean(results), mse, len(results) =  0.6564 tf.Tensor(418.45328, shape=(), dtype=float32) 900
slice = test, score = 0.6564

=== Iteration 37000 ===
np.mean(results), mse, len(results) =  0.6211 tf.Tensor(427.34302, shape=(), dtype=float32) 100
slice = key, score = 0.6211
np.mean(results), mse, len(results) =  0.7212856428214106 tf.Tensor(428.43906, shape=(), dtype=float32) 1999
slice = train, score = 0.7212856428214106
np.mean(results), mse, len(results) =  0.6577111111111111 tf.Tensor(431.9792, shape=(), dtype=float32) 900
slice = test, score =


=== Iteration 54000 ===
np.mean(results), mse, len(results) =  0.6217999999999999 tf.Tensor(494.95593, shape=(), dtype=float32) 100
slice = key, score = 0.6217999999999999
np.mean(results), mse, len(results) =  0.7281740870435217 tf.Tensor(498.73404, shape=(), dtype=float32) 1999
slice = train, score = 0.7281740870435217
np.mean(results), mse, len(results) =  0.6577222222222223 tf.Tensor(505.22247, shape=(), dtype=float32) 900
slice = test, score = 0.6577222222222223

=== Iteration 55000 ===
np.mean(results), mse, len(results) =  0.6153 tf.Tensor(504.87677, shape=(), dtype=float32) 100
slice = key, score = 0.6153
np.mean(results), mse, len(results) =  0.727263631815908 tf.Tensor(507.85623, shape=(), dtype=float32) 1999
slice = train, score = 0.727263631815908
np.mean(results), mse, len(results) =  0.6571111111111112 tf.Tensor(515.14154, shape=(), dtype=float32) 900
slice = test, score = 0.6571111111111112

=== Iteration 56000 ===
np.mean(results), mse, len(results) =  0.6182 tf.Tensor

np.mean(results), mse, len(results) =  0.7314207103551776 tf.Tensor(561.03406, shape=(), dtype=float32) 1999
slice = train, score = 0.7314207103551776
np.mean(results), mse, len(results) =  0.6576444444444445 tf.Tensor(568.74115, shape=(), dtype=float32) 900
slice = test, score = 0.6576444444444445

=== Iteration 73000 ===
np.mean(results), mse, len(results) =  0.6203 tf.Tensor(560.1028, shape=(), dtype=float32) 100
slice = key, score = 0.6203
np.mean(results), mse, len(results) =  0.7317208604302151 tf.Tensor(565.65344, shape=(), dtype=float32) 1999
slice = train, score = 0.7317208604302151
np.mean(results), mse, len(results) =  0.6570111111111111 tf.Tensor(574.2375, shape=(), dtype=float32) 900
slice = test, score = 0.6570111111111111

=== Iteration 74000 ===
np.mean(results), mse, len(results) =  0.6204000000000001 tf.Tensor(567.1495, shape=(), dtype=float32) 100
slice = key, score = 0.6204000000000001
np.mean(results), mse, len(results) =  0.7320810405202601 tf.Tensor(574.38074, sh

np.mean(results), mse, len(results) =  0.7346673336668333 tf.Tensor(636.1378, shape=(), dtype=float32) 1999
slice = train, score = 0.7346673336668333
np.mean(results), mse, len(results) =  0.6564222222222222 tf.Tensor(647.334, shape=(), dtype=float32) 900
slice = test, score = 0.6564222222222222

=== Iteration 91000 ===
np.mean(results), mse, len(results) =  0.6214 tf.Tensor(630.4782, shape=(), dtype=float32) 100
slice = key, score = 0.6214
np.mean(results), mse, len(results) =  0.7351225612806405 tf.Tensor(636.4811, shape=(), dtype=float32) 1999
slice = train, score = 0.7351225612806405
np.mean(results), mse, len(results) =  0.6558555555555555 tf.Tensor(647.1709, shape=(), dtype=float32) 900
slice = test, score = 0.6558555555555555

=== Iteration 92000 ===
np.mean(results), mse, len(results) =  0.6228999999999999 tf.Tensor(636.62164, shape=(), dtype=float32) 100
slice = key, score = 0.6228999999999999
np.mean(results), mse, len(results) =  0.734327163581791 tf.Tensor(643.95746, shape=

(0.736128064032016, 0.6564444444444444)

Popular -> 0.009166666666666667
AnnCur -> 0.6164111111111111
key_games=np.arange(100) -> 0.5720444444444445
K_Random -> 0.6130444444444445
K_KMeans -> 0.6123999999999999
K_BisectingKMeans -> 0.5866888888888888
K_MiniBatchKMeans -> 0.6082333333333333
K_AgglomerativeClustering -> 0.6150888888888889
K_SpectralCoclustering -> 0.5847111111111111
K_SpectralBiclustering -> 0.5804444444444445
K_SpectralClusteringNN -> 0.6133666666666666
K_ByMin -> 0.6282444444444444

K_Random + RBE :0.6250
K_KMeans : 0.6123999999999999 ->  0.6204777777777777
CoItem : 0.6458777777777778 -> 0.6564444444444444

In [ ]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

In [42]:
ctx.set_kmeans_games_as_key()

In [43]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1999, 100)
self.embed_games.shape =  (24493, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 24493)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1999, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0025 tf.Tensor(40.47996, shape=(), dtype=float32) 100
slice = key, score = 0.0025
np.mean(results), mse, len(results) =  0.003366683341670836 tf.Tensor(42.613895, shape=(), dtype=float32) 1999
slice = train, score = 0.003366683341670836
np.mean(results), mse, len(results) =  0.003588888888888889 tf.Tensor(42.73112, shape=(), dtype=float32) 900
slice = test, score = 0.003588888888888889

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5293000000000001 tf.Tensor(53.58917, shape=(), dtype=float32) 100
slice = key, score = 0.5293000000000001
np.mean(results), mse, len(results) =  0.6102701350675338 tf.Tensor(54.408394, shape=(), dtype=float32) 1999
slice = train

np.mean(results), mse, len(results) =  0.6821760880440221 tf.Tensor(304.8091, shape=(), dtype=float32) 1999
slice = train, score = 0.6821760880440221
np.mean(results), mse, len(results) =  0.6185666666666667 tf.Tensor(305.29004, shape=(), dtype=float32) 900
slice = test, score = 0.6185666666666667

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.5776 tf.Tensor(319.63025, shape=(), dtype=float32) 100
slice = key, score = 0.5776
np.mean(results), mse, len(results) =  0.6828014007003502 tf.Tensor(312.60742, shape=(), dtype=float32) 1999
slice = train, score = 0.6828014007003502
np.mean(results), mse, len(results) =  0.6186777777777778 tf.Tensor(313.52145, shape=(), dtype=float32) 900
slice = test, score = 0.6186777777777778

=== Iteration 19000 ===
np.mean(results), mse, len(results) =  0.5772000000000002 tf.Tensor(333.6008, shape=(), dtype=float32) 100
slice = key, score = 0.5772000000000002
np.mean(results), mse, len(results) =  0.6848974487243622 tf.Tensor(323.36813, s

np.mean(results), mse, len(results) =  0.6937568784392197 tf.Tensor(444.73233, shape=(), dtype=float32) 1999
slice = train, score = 0.6937568784392197
np.mean(results), mse, len(results) =  0.6222 tf.Tensor(445.8497, shape=(), dtype=float32) 900
slice = test, score = 0.6222

=== Iteration 36000 ===
np.mean(results), mse, len(results) =  0.5829000000000001 tf.Tensor(459.29816, shape=(), dtype=float32) 100
slice = key, score = 0.5829000000000001
np.mean(results), mse, len(results) =  0.6949174587293646 tf.Tensor(450.2734, shape=(), dtype=float32) 1999
slice = train, score = 0.6949174587293646
np.mean(results), mse, len(results) =  0.6196555555555556 tf.Tensor(451.4737, shape=(), dtype=float32) 900
slice = test, score = 0.6196555555555556

=== Iteration 37000 ===
np.mean(results), mse, len(results) =  0.5790000000000001 tf.Tensor(458.1656, shape=(), dtype=float32) 100
slice = key, score = 0.5790000000000001
np.mean(results), mse, len(results) =  0.6954177088544271 tf.Tensor(452.47446, sha

np.mean(results), mse, len(results) =  0.7007303651825912 tf.Tensor(531.9024, shape=(), dtype=float32) 1999
slice = train, score = 0.7007303651825912
np.mean(results), mse, len(results) =  0.6208 tf.Tensor(533.368, shape=(), dtype=float32) 900
slice = test, score = 0.6208

=== Iteration 54000 ===
np.mean(results), mse, len(results) =  0.5781000000000001 tf.Tensor(534.2563, shape=(), dtype=float32) 100
slice = key, score = 0.5781000000000001
np.mean(results), mse, len(results) =  0.7014057028514258 tf.Tensor(528.398, shape=(), dtype=float32) 1999
slice = train, score = 0.7014057028514258
np.mean(results), mse, len(results) =  0.6205444444444445 tf.Tensor(529.5272, shape=(), dtype=float32) 900
slice = test, score = 0.6205444444444445

=== Iteration 55000 ===
np.mean(results), mse, len(results) =  0.5784999999999999 tf.Tensor(539.10474, shape=(), dtype=float32) 100
slice = key, score = 0.5784999999999999
np.mean(results), mse, len(results) =  0.7022311155577788 tf.Tensor(532.68243, shape=

np.mean(results), mse, len(results) =  0.6218555555555556 tf.Tensor(603.31824, shape=(), dtype=float32) 900
slice = test, score = 0.6218555555555556

=== Iteration 72000 ===
np.mean(results), mse, len(results) =  0.5849 tf.Tensor(605.64764, shape=(), dtype=float32) 100
slice = key, score = 0.5849
np.mean(results), mse, len(results) =  0.7060580290145072 tf.Tensor(604.3763, shape=(), dtype=float32) 1999
slice = train, score = 0.7060580290145072
np.mean(results), mse, len(results) =  0.6208444444444444 tf.Tensor(604.3136, shape=(), dtype=float32) 900
slice = test, score = 0.6208444444444444

=== Iteration 73000 ===
np.mean(results), mse, len(results) =  0.5858 tf.Tensor(612.04114, shape=(), dtype=float32) 100
slice = key, score = 0.5858
np.mean(results), mse, len(results) =  0.7063531765882941 tf.Tensor(610.7434, shape=(), dtype=float32) 1999
slice = train, score = 0.7063531765882941
np.mean(results), mse, len(results) =  0.6204777777777777 tf.Tensor(611.9855, shape=(), dtype=float32) 90


=== Iteration 90000 ===
np.mean(results), mse, len(results) =  0.5825 tf.Tensor(681.743, shape=(), dtype=float32) 100
slice = key, score = 0.5825
np.mean(results), mse, len(results) =  0.71016008004002 tf.Tensor(678.0829, shape=(), dtype=float32) 1999
slice = train, score = 0.71016008004002
np.mean(results), mse, len(results) =  0.6212000000000001 tf.Tensor(678.90533, shape=(), dtype=float32) 900
slice = test, score = 0.6212000000000001

=== Iteration 91000 ===
np.mean(results), mse, len(results) =  0.5823 tf.Tensor(683.2457, shape=(), dtype=float32) 100
slice = key, score = 0.5823
np.mean(results), mse, len(results) =  0.7105152576288145 tf.Tensor(681.07, shape=(), dtype=float32) 1999
slice = train, score = 0.7105152576288145
np.mean(results), mse, len(results) =  0.6209111111111112 tf.Tensor(680.5882, shape=(), dtype=float32) 900
slice = test, score = 0.6209111111111112

=== Iteration 92000 ===
np.mean(results), mse, len(results) =  0.5756 tf.Tensor(689.6585, shape=(), dtype=float32

(0.7101850925462732, 0.6191444444444445)

In [44]:
ctx.key_games = np.random.choice(np.arange(ctx.games_count), ctx.key_size, replace=False)

In [45]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1999, 100)
self.embed_games.shape =  (24493, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 24493)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1999, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0029000000000000002 tf.Tensor(38.31489, shape=(), dtype=float32) 100
slice = key, score = 0.0029000000000000002
np.mean(results), mse, len(results) =  0.005142571285642822 tf.Tensor(37.841095, shape=(), dtype=float32) 1999
slice = train, score = 0.005142571285642822
np.mean(results), mse, len(results) =  0.0037333333333333337 tf.Tensor(37.865456, shape=(), dtype=float32) 900
slice = test, score = 0.0037333333333333337

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5466000000000001 tf.Tensor(51.295834, shape=(), dtype=float32) 100
slice = key, score = 0.5466000000000001
np.mean(results), mse, len(results) =  0.6161480740370185 tf.Tensor(53.055893, shape=(),

np.mean(results), mse, len(results) =  0.6902851425712856 tf.Tensor(303.21515, shape=(), dtype=float32) 1999
slice = train, score = 0.6902851425712856
np.mean(results), mse, len(results) =  0.6231333333333333 tf.Tensor(307.59198, shape=(), dtype=float32) 900
slice = test, score = 0.6231333333333333

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.6007000000000001 tf.Tensor(310.74814, shape=(), dtype=float32) 100
slice = key, score = 0.6007000000000001
np.mean(results), mse, len(results) =  0.6909854927463732 tf.Tensor(315.59665, shape=(), dtype=float32) 1999
slice = train, score = 0.6909854927463732
np.mean(results), mse, len(results) =  0.6236555555555555 tf.Tensor(320.10052, shape=(), dtype=float32) 900
slice = test, score = 0.6236555555555555

=== Iteration 19000 ===
np.mean(results), mse, len(results) =  0.598 tf.Tensor(319.99103, shape=(), dtype=float32) 100
slice = key, score = 0.598
np.mean(results), mse, len(results) =  0.6919959979989995 tf.Tensor(325.6205, sh

np.mean(results), mse, len(results) =  0.6236666666666666 tf.Tensor(438.43817, shape=(), dtype=float32) 900
slice = test, score = 0.6236666666666666

=== Iteration 36000 ===
np.mean(results), mse, len(results) =  0.5973 tf.Tensor(425.03436, shape=(), dtype=float32) 100
slice = key, score = 0.5973
np.mean(results), mse, len(results) =  0.704112056028014 tf.Tensor(432.186, shape=(), dtype=float32) 1999
slice = train, score = 0.704112056028014
np.mean(results), mse, len(results) =  0.6254111111111111 tf.Tensor(438.95386, shape=(), dtype=float32) 900
slice = test, score = 0.6254111111111111

=== Iteration 37000 ===
np.mean(results), mse, len(results) =  0.5961 tf.Tensor(437.49792, shape=(), dtype=float32) 100
slice = key, score = 0.5961
np.mean(results), mse, len(results) =  0.7033016508254127 tf.Tensor(441.19547, shape=(), dtype=float32) 1999
slice = train, score = 0.7033016508254127
np.mean(results), mse, len(results) =  0.6231333333333334 tf.Tensor(448.3536, shape=(), dtype=float32) 900


=== Iteration 54000 ===
np.mean(results), mse, len(results) =  0.5921 tf.Tensor(520.27045, shape=(), dtype=float32) 100
slice = key, score = 0.5921
np.mean(results), mse, len(results) =  0.7091945972986493 tf.Tensor(524.4802, shape=(), dtype=float32) 1999
slice = train, score = 0.7091945972986493
np.mean(results), mse, len(results) =  0.6224888888888889 tf.Tensor(533.34485, shape=(), dtype=float32) 900
slice = test, score = 0.6224888888888889

=== Iteration 55000 ===
np.mean(results), mse, len(results) =  0.5968 tf.Tensor(527.9664, shape=(), dtype=float32) 100
slice = key, score = 0.5968
np.mean(results), mse, len(results) =  0.7095697848924462 tf.Tensor(531.3875, shape=(), dtype=float32) 1999
slice = train, score = 0.7095697848924462
np.mean(results), mse, len(results) =  0.6234666666666667 tf.Tensor(541.15, shape=(), dtype=float32) 900
slice = test, score = 0.6234666666666667

=== Iteration 56000 ===
np.mean(results), mse, len(results) =  0.5957 tf.Tensor(532.022, shape=(), dtype=fl


=== Iteration 72000 ===
np.mean(results), mse, len(results) =  0.5980000000000001 tf.Tensor(614.19836, shape=(), dtype=float32) 100
slice = key, score = 0.5980000000000001
np.mean(results), mse, len(results) =  0.7132866433216609 tf.Tensor(612.3218, shape=(), dtype=float32) 1999
slice = train, score = 0.7132866433216609
np.mean(results), mse, len(results) =  0.6213222222222223 tf.Tensor(622.3957, shape=(), dtype=float32) 900
slice = test, score = 0.6213222222222223

=== Iteration 73000 ===
np.mean(results), mse, len(results) =  0.5941000000000001 tf.Tensor(618.7115, shape=(), dtype=float32) 100
slice = key, score = 0.5941000000000001
np.mean(results), mse, len(results) =  0.714767383691846 tf.Tensor(622.99506, shape=(), dtype=float32) 1999
slice = train, score = 0.714767383691846
np.mean(results), mse, len(results) =  0.6217333333333332 tf.Tensor(633.7224, shape=(), dtype=float32) 900
slice = test, score = 0.6217333333333332

=== Iteration 74000 ===
np.mean(results), mse, len(results)

np.mean(results), mse, len(results) =  0.7173486743371686 tf.Tensor(724.20526, shape=(), dtype=float32) 1999
slice = train, score = 0.7173486743371686
np.mean(results), mse, len(results) =  0.6201555555555557 tf.Tensor(737.41943, shape=(), dtype=float32) 900
slice = test, score = 0.6201555555555557

=== Iteration 92000 ===
np.mean(results), mse, len(results) =  0.5925 tf.Tensor(734.70447, shape=(), dtype=float32) 100
slice = key, score = 0.5925
np.mean(results), mse, len(results) =  0.7170535267633817 tf.Tensor(734.5184, shape=(), dtype=float32) 1999
slice = train, score = 0.7170535267633817
np.mean(results), mse, len(results) =  0.6208777777777778 tf.Tensor(748.4055, shape=(), dtype=float32) 900
slice = test, score = 0.6208777777777778

=== Iteration 93000 ===
np.mean(results), mse, len(results) =  0.5941000000000001 tf.Tensor(731.46094, shape=(), dtype=float32) 100
slice = key, score = 0.5941000000000001
np.mean(results), mse, len(results) =  0.7175387693846924 tf.Tensor(739.2626, sh

(0.7183041520760379, 0.6200999999999999)